In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/sample_output.csv
/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/custom_archive.py
/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/embeddings.npy
/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/subtask1.npy
/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/subtask2.npy


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [3]:
embeds= np.load("/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/embeddings.npy")
embeds_1= np.load("/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/subtask1.npy")
embeds_2= np.load("/kaggle/input/datasets/murtazaabdullah2010/roai-selection-camp-cpu-1/train_data/subtask2.npy")

In [4]:
print(len(embeds), len(embeds_1), len(embeds_2))
print(embeds.shape, embeds_1.shape, embeds_2.shape)

4000 4000 4000
(4000, 384) (4000, 768) (4000, 768)


In [12]:
#subtask 1 
sub1_x = np.concatenate([embeds[:400] , embeds[:400]], axis = 0)
sub1_x.shape

(800, 384)

In [15]:
embeds_1[:800]

array([[ 0.03438123,  0.00717025, -0.01272174, ...,  0.03176759,
        -0.04366959,  0.00736909],
       [ 0.00868411,  0.0434986 ,  0.01925925, ...,  0.02864279,
         0.01797144, -0.02536913],
       [ 0.004288  , -0.06158917, -0.01655139, ..., -0.0584551 ,
         0.01024327,  0.00324681],
       ...,
       [ 0.01093752, -0.06136505, -0.02254017, ..., -0.00808088,
        -0.02751576,  0.02570631],
       [-0.01452831,  0.03849414,  0.02520507, ...,  0.00951827,
        -0.03265129,  0.00282118],
       [-0.00552375,  0.00994683,  0.00952291, ..., -0.01011632,
         0.01344206,  0.04565948]], shape=(800, 768))

In [27]:
class DF(nn.Module):
    def __init__(self, embeds, sub_embeds):
        self.embeds = embeds
        self.sub_embeds = sub_embeds
    def __len__(self):
        return len(self.embeds)
    def __getitem__(self, idx):
        embed= self.embeds[idx]
        sub_embed = self.sub_embeds[idx]
        if idx <400:
            y = torch.tensor(1.0, dtype = torch.float32)
        else:
            y = torch.tensor(0.0, dtype = torch.float32)
        return torch.tensor(embed, dtype =torch.float32), torch.tensor(sub_embed, dtype = torch.float32), y
train_df = DF(sub1_x, embeds_1[:800])
train_dl = DataLoader(train_df, batch_size= 16, shuffle = True, num_workers =4 , pin_memory = True)

for batch in train_dl:
    x,x1, y= batch
    print(x.shape)
    print(x1.shape)
    print(y.shape)
    break

torch.Size([16, 384])
torch.Size([16, 768])
torch.Size([16])


In [28]:
class Block(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(i, o),
            nn.LayerNorm(o),
            nn.GELU(),
            nn.Dropout(0.1)
        )
    def forward(self, x):
        return self.block(x)
class SNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj_1 = nn.Linear(384, 512)
        self.proj_2 = nn.Linear(768, 512)  
        self.model = nn.Sequential(
            Block(512, 1024),
            Block(1024, 2048),
            Block(2048, 1024),
            Block(1024, 512)
        )
        self.head = nn.Sequential(
            Block(1024, 512),
            nn.Linear(512, 1) 
        )
    def forward(self, x1, x2):
        x1 = self.model(self.proj_1(x1))
        x2 = self.model(self.proj_2(x2))
        x = torch.cat([x1, x2], dim=1) 
        return self.head(x)
device = ("cuda" if torch.cuda.is_available() else "cpu")
model = SNN().to(device)

In [30]:
opt = torch.optim.AdamW(model.parameters(), lr =1e-5,weight_decay =2e-5)
loss_fn = nn.BCEWithLogitsLoss()

for e in range(10):
    t_loss= 0
    model.train()
    for batch in train_dl:
        x1, x2, y = batch
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        out= model(x1, x2)
        y = y.unsqueeze(1)
        loss= loss_fn(out, y)
        opt.zero_grad()
        loss.backward()
        t_loss += loss.item()
        opt.step()
    print("E: ", e, "train_loss: ", t_loss/len(train_dl))

E:  0 train_loss:  0.702843245267868
E:  1 train_loss:  0.6939331734180451
E:  2 train_loss:  0.6931093668937683
E:  3 train_loss:  0.6954110205173493
E:  4 train_loss:  0.6831882870197297
E:  5 train_loss:  0.667679454088211
E:  6 train_loss:  0.6618010437488556
E:  7 train_loss:  0.6405142045021057
E:  8 train_loss:  0.6369836789369583
E:  9 train_loss:  0.6156766790151597


In [ ]:
test_preds = 